# 01 | How radar responds to soil and vegetation
Sentinel-1 sends C-band microwaves (~5.4 GHz) and measures backscatter. Water in soil changes dielectric properties, but roughness, vegetation, incidence angle, and standing water also change the returned signal. VV is co-polarized; VH is cross-polarized. Higher VV does not uniquely imply wetter soil.

The following **illustrative** equations are not a calibrated scattering model. Fresnel reflection at a planar interface differs from distributed SAR backscatter. The examples teach ambiguity, not how to estimate absolute moisture from one pixel.

## Setup (CPU, standard RAM)
Run this cell first in each notebook. The project uses small monthly raster tiles and tabular ML. If Colab requests a restart, use Runtime → Restart session and rerun from the beginning. Do not change NumPy manually. Files persist in Drive.

In [ ]:
%pip -q install earthengine-api rasterio pyproj scikit-learn joblib
import sys,json
from pathlib import Path
from importlib.metadata import version
for pkg,mod in [('numpy','numpy'),('scikit-learn','sklearn'),('rasterio','rasterio')]:
    loaded=sys.modules.get(mod)
    if loaded is not None and getattr(loaded,'__version__',version(pkg))!=version(pkg):
        raise RuntimeError(f'{pkg} changed. Restart session and rerun.')
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Bagre_SAR_Soil_Moisture')
assert (ROOT/'code'/'sar_workflow.py').is_file(), 'Check Drive folder and shared code.'
sys.path.insert(0,str(ROOT/'code'))
import sar_workflow as sw
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
for sub in ['data','models','outputs','figures','qgis']: (ROOT/sub).mkdir(exist_ok=True)
print('NumPy:',version('numpy'),'scikit-learn:',version('scikit-learn'))

## Decibels versus linear power
Backscatter is commonly written as σ⁰ in linear power or 10 log₁₀(σ⁰) in dB. Arithmetic averages must be computed in power space, then converted to dB for visualization.

In [ ]:
values=np.array([-20.,-10.]); p=sw.power(values)
print('dB:',values,'power:',p)
print('Mean of dB:',values.mean(),'dB; dB of mean power:',sw.db(p.mean()),'dB')
assert np.allclose(sw.db(sw.power(values)),values)

## Dielectric contrast is only one factor
Use relative permittivity values as an illustrative axis, not measured soil moisture. The normal-incidence power reflection coefficient is R = ((√ε−1)/(√ε+1))². A rough surface, unlike an ideal flat interface, can scatter energy back to the radar. Vegetation attenuates the soil return and scatters independently.

In [ ]:
eps=np.linspace(3,35,150); refl=((np.sqrt(eps)-1)/(np.sqrt(eps)+1))**2
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(eps,refl); ax.set(xlabel='Illustrative relative permittivity',ylabel='Normal-incidence power reflection',title='Dielectric contrast (not SAR backscatter)'); ax.grid(alpha=.3)
fig.savefig(ROOT/'figures'/'01_dielectric.png',dpi=150,bbox_inches='tight'); plt.show()

## A toy canopy attenuation example
For illustration, σ⁰ = A exp(−2τ/cos θ) + B(1−exp(−2τ/cos θ)). A represents an assumed soil-scattering term, B an assumed vegetation-scattering term, τ an optical-depth-like parameter. This is a simplified teaching expression, not a fitted Water Cloud Model and not an inversion. Use the plot to explain why vegetation and roughness confound soil moisture.

In [ ]:
tau=np.linspace(0,1.2,150); theta=np.deg2rad(35); transmission=np.exp(-2*tau/np.cos(theta))
fig,ax=plt.subplots(figsize=(7,4))
for soil in [.02,.05,.10]:
    signal=soil*transmission+.035*(1-transmission)
    ax.plot(tau,sw.db(signal),label=f'Assumed soil power {soil}')
ax.set(xlabel='Illustrative vegetation optical depth',ylabel='Toy signal (dB)',title='Different soil signals converge under vegetation'); ax.legend(); ax.grid(alpha=.3)
fig.savefig(ROOT/'figures'/'01_toy_canopy.png',dpi=150,bbox_inches='tight'); plt.show()

**Checkpoint:** explain why radar alone cannot uniquely determine moisture, why averaging in dB is incorrect for mean power, and why the toy equation is not a retrieval model. Sources: [Sentinel-1 Earth Engine guide](https://developers.google.com/earth-engine/guides/sentinel1), [SMAP product](https://developers.google.com/earth-engine/datasets/catalog/NASA_SMAP_SPL3SMP_E_006).